In [2]:
!pip install torch

import torch
import torch.nn as nn
import torch.optim as optim
import math


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

Using Device: cuda


In [4]:
# POSITIONAL ENCODING
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):

        x = x + self.pe[:, :x.size(1)]

        return x

In [8]:
# SCALED DOT PRODUCT ATTENTION
class ScaledDotProductAttention(nn.Module):

    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention_weights, V)

        return output, attention_weights


In [9]:
# MULTI HEAD ATTENTION
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def split_heads(self, x, batch_size):

        x = x.view(batch_size, -1, self.num_heads, self.d_k)

        return x.transpose(1, 2)

    def forward(self, Q, K, V, mask=None):

        batch_size = Q.size(0)

        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)

        attention_output, attention_weights = self.attention(
            Q, K, V, mask
        )

        attention_output = attention_output.transpose(1, 2).contiguous()

        attention_output = attention_output.view(
            batch_size,
            -1,
            self.d_model
        )

        output = self.fc(attention_output)

        return output


In [10]:
# FEED FORWARD NETWORK
class FeedForwardNetwork(nn.Module):

    def __init__(self, d_model, d_ff):
        super(FeedForwardNetwork, self).__init__()

        self.linear1 = nn.Linear(d_model, d_ff)

        self.relu = nn.ReLU()

        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):

        x = self.linear1(x)

        x = self.relu(x)

        x = self.linear2(x)

        return x


In [11]:
# ENCODER LAYER
class EncoderLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)

        self.ffn = FeedForwardNetwork(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)

        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        attention_output = self.mha(x, x, x, mask)

        x = self.norm1(x + self.dropout(attention_output))

        ffn_output = self.ffn(x)

        x = self.norm2(x + self.dropout(ffn_output))

        return x

In [12]:
# TRANSFORMER MODEL
class Transformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        max_len,
        dropout=0.1
    ):

        super(Transformer, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.positional_encoding = PositionalEncoding(
            d_model,
            max_len
        )

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        x = self.embedding(x)

        x = self.positional_encoding(x)

        x = self.dropout(x)

        for layer in self.layers:

            x = layer(x, mask)

        output = self.fc_out(x)

        return output


In [13]:
# HYPERPARAMETERS
VOCAB_SIZE = 1000
D_MODEL = 128
NUM_HEADS = 8
NUM_LAYERS = 4
D_FF = 512
MAX_LEN = 50
DROPOUT = 0.1

BATCH_SIZE = 16
SEQ_LENGTH = 20

EPOCHS = 10
LEARNING_RATE = 0.001


In [14]:
# MODEL INITIALIZATION
model = Transformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT
).to(device)

print(model)


Transformer(
  (embedding): Embedding(1000, 128)
  (positional_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (mha): MultiHeadAttention(
        (W_q): Linear(in_features=128, out_features=128, bias=True)
        (W_k): Linear(in_features=128, out_features=128, bias=True)
        (W_v): Linear(in_features=128, out_features=128, bias=True)
        (fc): Linear(in_features=128, out_features=128, bias=True)
        (attention): ScaledDotProductAttention()
      )
      (ffn): FeedForwardNetwork(
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (relu): ReLU()
        (linear2): Linear(in_features=512, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (fc_out): Linear(in_features=128, out_features=1000, bias=True)
  (dropout): Dropout(p

In [15]:
# LOSS FUNCTION & OPTIMIZER
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [16]:
# DUMMY DATASET
# Random input token IDs
input_data = torch.randint(
    0,
    VOCAB_SIZE,
    (BATCH_SIZE, SEQ_LENGTH)
).to(device)

# Random target token IDs
target_data = torch.randint(
    0,
    VOCAB_SIZE,
    (BATCH_SIZE, SEQ_LENGTH)
).to(device)


In [17]:
# TRAINING LOOP
print("\nStarting Training...\n")

for epoch in range(EPOCHS):

    model.train()

    optimizer.zero_grad()

    outputs = model(input_data)

    outputs = outputs.view(-1, VOCAB_SIZE)

    targets = target_data.view(-1)

    loss = criterion(outputs, targets)

    loss.backward()

    optimizer.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Loss: {loss.item():.4f}")
    print("-" * 40)

print("\nTraining Completed Successfully!")



Starting Training...

Epoch [1/10]
Loss: 7.0081
----------------------------------------
Epoch [2/10]
Loss: 6.6264
----------------------------------------
Epoch [3/10]
Loss: 6.3417
----------------------------------------
Epoch [4/10]
Loss: 6.0845
----------------------------------------
Epoch [5/10]
Loss: 5.8345
----------------------------------------
Epoch [6/10]
Loss: 5.5158
----------------------------------------
Epoch [7/10]
Loss: 5.2477
----------------------------------------
Epoch [8/10]
Loss: 4.9888
----------------------------------------
Epoch [9/10]
Loss: 4.7358
----------------------------------------
Epoch [10/10]
Loss: 4.5089
----------------------------------------

Training Completed Successfully!


In [18]:
# TESTING
model.eval()

with torch.no_grad():

    sample_input = torch.randint(
        0,
        VOCAB_SIZE,
        (1, SEQ_LENGTH)
    ).to(device)

    prediction = model(sample_input)

    predicted_tokens = torch.argmax(prediction, dim=-1)

    print("\nSample Prediction:")
    print(predicted_tokens)


Sample Prediction:
tensor([[555, 957, 457, 328, 940, 889, 116, 120, 735, 817, 116, 682, 785, 456,
         735, 616, 144, 317, 166, 898]], device='cuda:0')


In [19]:

# SAVE MODEL
torch.save(model.state_dict(), "transformer_model.pth")

print("\nModel Saved Successfully!")


Model Saved Successfully!
